In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# 1. Inicialização da Sessão Spark
spark = SparkSession.builder.appName("Hadoop_Ecossistema_Vendas").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# 2. Definição do Esquema Estruturado (StructType)
esquema_vendas = StructType([
    StructField('id_venda', IntegerType()),
    StructField('data_venda', StringType()),
    StructField('hora_venda', StringType()),
    StructField('id_produto', IntegerType()),
    StructField('produto', StringType()),
    StructField('categoria', StringType()),
    StructField('cor', StringType()),
    StructField('tamanho', StringType()),
    StructField('quantidade', IntegerType()),
    StructField('preco_unitario', DoubleType()),
    StructField('desconto', DoubleType()),
    StructField('valor_total', StringType()),
    StructField('forma_pagamento', StringType()),
    StructField('canal', StringType()),
    StructField('id_cliente', IntegerType()),
    StructField('id_vendedor', IntegerType())
])

In [ ]:
# 3. Leitura Streaming / Batch com o Esquema Definido
df_spark_vendas = spark.read.schema(esquema_vendas).option("header", "true").csv("../vendas.csv")

In [ ]:
# 4. Transformação dos Dados: Criação de coluna de classificação condicional
df_situacao_vendas = df_spark_vendas.withColumn(
    "classificacao_desconto",
    F.when(F.col("desconto") > 0, "Com Desconto").otherwise("Sem Desconto")
)

In [ ]:
# 5. Registro de Tabela Temporária para Consultas SQL
df_situacao_vendas.createOrReplaceTempView("vendas_stream")

In [ ]:
# 6. Execução de Consulta SQL no Spark
resultado_spark_sql = spark.sql("""
    SELECT
        categoria,
        classificacao_desconto,
        COUNT(*) AS total_transacoes,
        SUM(quantidade) AS total_itens_vendidos
    FROM vendas_stream
    GROUP BY categoria, classificacao_desconto
    ORDER BY total_itens_vendidos DESC
""")

resultado_spark_sql.show(10, truncate=False)

In [ ]:
# Encerramento dos recursos do Spark
spark.stop()